# BÀI LAB THỰC HÀNH SQL VỚI SQLITE3 TRONG JUPYTER NOTEBOOK

## Mục tiêu bài lab
Sau khi hoàn thành bài lab này, bạn sẽ nắm vững cách sử dụng các câu lệnh SQL thuộc các nhóm:

- DDL (Data Definition Language): CREATE, ALTER, DROP
- DML (Data Manipulation Language): INSERT, UPDATE, DELETE
- DQL (Data Query Language): SELECT kết hợp với các mệnh đề WHERE, ORDER BY, LIMIT, v.v.
- Joins & Subqueries: INNER JOIN, LEFT JOIN, truy vấn con.
- Advanced: GROUP BY, HAVING, Hàm tổng hợp, UNION, VIEW, INDEX, Transaction.

## Phần 0: Khởi tạo môi trường

Chạy cell dưới đây để kết nối với database SQLite và tạo một hàm hỗ trợ hiển thị kết quả dưới dạng bảng (giống pandas) cho đẹp mắt.

In [2]:
import sqlite3
import pandas as pd

In [3]:
# Kết nối đến database (file 'lab_sql.db' sẽ được tạo ở thư mục hiện tại)
conn = sqlite3.connect('lab_sql.db')
cursor = conn.cursor()

In [4]:
# Hàm hỗ trợ chạy SQL và hiển thị kết quả dạng bảng đẹp mắt
def run_sql(sql):
    try:
        df = pd.read_sql_query(sql, conn)
        display(df)
    except Exception as e:
        print(f"Lỗi: {e}")

print("Kết nối thành công đến database!")

Kết nối thành công đến database!


## PHẦN 1: DDL (Data Definition Language)
Ngôn ngữ định nghĩa dữ liệu: Dùng để tạo, sửa đổi và xóa cấu trúc bảng.

### 1.1. Lệnh CREATE TABLE
Chúng ta sẽ tạo 2 bảng: Departments (Phòng ban) và Employees (Nhân viên).

In [8]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS Departments (
    DeptID INTEGER PRIMARY KEY AUTOINCREMENT,
    DeptName TEXT NOT NULL,
    Location TEXT
);
''')

In [9]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS Employees (
    EmpID INTEGER PRIMARY KEY AUTOINCREMENT,
    Name TEXT NOT NULL,
    Age INTEGER,
    Salary REAL,
    DeptID INTEGER,
    ManagerID INTEGER,
    HireDate TEXT,
    FOREIGN KEY (DeptID) REFERENCES Departments(DeptID)
);
''')
conn.commit()
print("Đã tạo bảng Departments và Employees thành công!")

Đã tạo bảng Departments và Employees thành công!


#### TODO 1: Tạo bảng mới
- Yêu cầu: Tạo một bảng tên là Projects có các cột: ProjectID (Khóa chính, tự tăng), ProjectName (Text, Not Null), Budget (Real), DeptID (Khóa ngoại tham chiếu đến bảng Departments).

In [10]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS Projects (
    ProjectID INTEGER PRIMARY KEY AUTOINCREMENT,
    ProjectName TEXT NOT NULL,
    Budget REAL,
    DeptID INTEGER,
    FOREIGN KEY (DeptID) REFERENCES Departments(DeptID)
);
''')
conn.commit()
print("Đã tạo bảng Projects thành công!")

Đã tạo bảng Projects thành công!


### 1.2. Lệnh ALTER TABLE
Dùng để thêm hoặc sửa đổi cột.

In [11]:
# Thêm cột mới vào bảng Employees
cursor.execute('ALTER TABLE Employees ADD COLUMN Email TEXT;')
conn.commit()
run_sql("PRAGMA table_info(Employees);") # Xem cấu trúc bảng

,cid,name,type,notnull,dflt_value,pk
0,0,EmpID,INTEGER,0,None,1
1,1,Name,TEXT,1,None,0
2,2,Age,INTEGER,0,None,0
3,3,Salary,REAL,0,None,0
4,4,DeptID,INTEGER,0,None,0
5,5,ManagerID,INTEGER,0,None,0
6,6,HireDate,TEXT,0,None,0
7,7,Email,TEXT,0,None,0


#### TODO 2: Sửa đổi bảng
Yêu cầu: Thêm một cột Status (kiểu TEXT, mặc định là 'Active') vào bảng Projects vừa tạo.

In [13]:
cursor.execute('''
ALTER TABLE Projects 
ADD COLUMN status TEXT DEFAULT 'active';
''')
conn.commit()
print("Đã thêm cột status thành công!")

Đã thêm cột status thành công!


In [14]:
run_sql("PRAGMA table_info(Projects);") # Xem cấu trúc bảng

,cid,name,type,notnull,dflt_value,pk
0,0,ProjectID,INTEGER,0,None,1
1,1,ProjectName,TEXT,1,None,0
2,2,Budget,REAL,0,None,0
3,3,DeptID,INTEGER,0,None,0
4,4,status,TEXT,0,'active',0


## PHẦN 2: DML (Data Manipulation Language)
Ngôn ngữ thao tác dữ liệu: Thêm, Sửa, Xóa dòng dữ liệu.

### 2.1. Lệnh INSERT
Chèn dữ liệu vào bảng.

In [16]:
# Insert Departments
cursor.executemany('''
INSERT INTO Departments (DeptName, Location) VALUES (?, ?)
''', [
    ('Engineering', 'Floor 1'),
    ('Marketing', 'Floor 2'),
    ('Sales', 'Floor 3'),
    ('HR', 'Floor 4')
])
conn.commit()

In [18]:
# Insert Employees
cursor.executemany('''
INSERT INTO Employees (Name, Age, Salary, DeptID, ManagerID, HireDate) 
VALUES (?, ?, ?, ?, ?, ?)
''', [
    ('Nguyen Van A', 28, 15000000, 1, 0, '2020-05-15'),
    ('Le Thi B', 32, 20000000, 1, 1, '2018-01-10'),
    ('Tran Van C', 25, 12000000, 2, 0, '2022-08-01'),
    ('Pham Thi D', 40, 25000000, 3, 0, '2015-11-20'),
    ('Hoang Van E', 30, 18000000, 2, 3, '2019-06-05'),
    ('Vo Thi F', 26, 11000000, 3, 4, '2023-01-15'),
    ('Bui Van G', 35, 30000000, 1, 2, '2012-03-25')
])
conn.commit()

print("Đã chèn dữ liệu mẫu!")

Đã chèn dữ liệu mẫu!


run_sql("SELECT * FROM Departments;")

#### TODO 3: Thêm dữ liệu
Yêu cầu: Thêm 2 dòng dữ liệu vào bảng Projects (Project1: 'Website Redesign', Budget: 50000000, DeptID: 1; Project2: 'Q4 Campaign', Budget: 20000000, DeptID: 2).